In [2]:
import numpy as np
import pandas as pd

In [3]:
def generate_synthetic_row():

    # 1. Loss streak
    loss_streak = np.random.choice(
        [0, 1, 2, 3, 4],
        p=[0.40, 0.25, 0.20, 0.10, 0.05]
    )

    # 2. Time between trades
    # Losses increase the chance of fast re-entry,
    # but do not guarantee it.
    if loss_streak >= 2:
        if np.random.rand() < 0.65:
            time_delta = np.random.exponential(scale=20)
        else:
            time_delta = np.random.exponential(scale=50)
    else:
        time_delta = np.random.exponential(scale=50)

    # 3. Change in position size
    # Losses increase the chance of increasing position size,
    # but normal sizing is still possible.
    if loss_streak >= 2:
        if np.random.rand() < 0.60:
            lot_mult = np.random.uniform(1.5, 3.5)
        else:
            lot_mult = np.random.uniform(0.8, 1.8)
    else:
        lot_mult = np.random.uniform(0.8, 1.8)

    # 4. Hedging behavior
    # Losses increase the chance of being unhedged,
    # but do not guarantee it.
    if loss_streak >= 2:
        is_unhedged = np.random.choice(
            [0, 1],
            p=[0.35, 0.65]
        )
    else:
        is_unhedged = np.random.choice(
            [0, 1],
            p=[0.50, 0.50]
        )

    # 5. Recent P&L
    # A loss streak means the recent trade was a loss,
    # but the loss can be small or large.
    if loss_streak >= 1:
        recent_pnl = -abs(
            np.random.normal(2500, 2500)
        )
    else:
        # With no loss streak, P&L can be positive or negative.
        recent_pnl = np.random.normal(
            1500, 2000
        )

    # --------------------------------------------------
    # 6. Weighted behavioral risk score
    # --------------------------------------------------

    loss_streak_risk = min(
        loss_streak / 3,
        1.0
    )

    # Faster re-entry = higher risk
    reentry_risk = max(
        0,
        1 - (time_delta / 120)
    )

    # Larger position = higher risk
    lotsize_risk = min(
        max(lot_mult - 1, 0) / 2,
        1.0
    )

    # Unhedged = additional risk
    unhedged_risk = float(is_unhedged)

    # Larger negative P&L = higher risk
    pnl_risk = min(
        max(-recent_pnl, 0) / 5000,
        1.0
    )

    # Weighted risk score
    risk_score = (
        0.30 * loss_streak_risk +
        0.20 * reentry_risk +
        0.20 * lotsize_risk +
        0.15 * unhedged_risk +
        0.15 * pnl_risk
    )

    # --------------------------------------------------
    # 7. Convert risk score into label
    # --------------------------------------------------

    label = 1 if risk_score > 0.75 else 0

    # --------------------------------------------------
    # 8. Add 5% noise
    # --------------------------------------------------

    if np.random.rand() < 0.02:
        label = 1 - label

    return [
        time_delta,
        lot_mult,
        is_unhedged,
        loss_streak,
        recent_pnl,
        label
    ]

In [4]:
# DATA GENERATION
data = [generate_synthetic_row() for _ in range(5000)]
df = pd.DataFrame(data, columns=[
    "time_delta_seconds","lot_size_multiplier","is_unhedged",
    "loss_streak_count","recent_pnl_delta","label"
])

In [5]:
df['label'].value_counts(normalize=True)

label
0    0.8768
1    0.1232
Name: proportion, dtype: float64

In [6]:
# --------------------------------------------------
# Basic checks
# --------------------------------------------------

print("Dataset shape:")
print(df.shape)

print("\nClass distribution:")
print(df["label"].value_counts())

print("\nClass distribution (%):")
print(df["label"].value_counts(normalize=True))

print("\nFeature statistics by label:")
print(
    df.groupby("label").mean(numeric_only=True)
)

print("\nMissing values:")
print(df.isnull().sum())

print("\nLoss streak vs label:")
print(
    pd.crosstab(
        df["loss_streak_count"],
        df["label"],
        normalize="index"
    )
)

Dataset shape:
(5000, 6)

Class distribution:
label
0    4384
1     616
Name: count, dtype: int64

Class distribution (%):
label
0    0.8768
1    0.1232
Name: proportion, dtype: float64

Feature statistics by label:
       time_delta_seconds  lot_size_multiplier  is_unhedged  \
label                                                         
0               46.691652             1.441925     0.509580   
1               21.101892             2.404795     0.863636   

       loss_streak_count  recent_pnl_delta  
label                                       
0               0.960538       -867.378916  
1               2.602273      -3280.623573  

Missing values:
time_delta_seconds     0
lot_size_multiplier    0
is_unhedged            0
loss_streak_count      0
recent_pnl_delta       0
label                  0
dtype: int64

Loss streak vs label:
label                     0         1
loss_streak_count                    
0                  0.980283  0.019717
1                  0.977165  0.022

In [8]:
print(df.columns.tolist())

['time_delta_seconds', 'lot_size_multiplier', 'is_unhedged', 'loss_streak_count', 'recent_pnl_delta', 'label']


NameError: name 'df' is not defined